# **Flood Susceptibility Mapping for Nairobi, Kenya**

Student: Esther Onyando  

Course: MSC Data Science  

Date: April 2026

Strathmore University 

## **Business Context**

Nairobi faces frequent, severe floods during rainy seasons that damage homes, roads, and businesses, and sometimes cause deaths. Flooding displaces families and increases health risks from contaminated water. Informal settlements and low-lying neighborhoods are most exposed because of poor drainage and dense development. Floods disrupt transport, damage critical infrastructure, and raise recovery costs for the city. Limited flood records and sparse monitoring make it hard for planners to predict where floods will occur. Current early warning and drainage planning systems are not enough for many vulnerable communities. This project builds a flood susceptibility map using satellite data and machine learning to show where floods are likely. The map and a reproducible workflow will help planners and emergency teams focus drainage, warnings, and response.

### Project Objectives

1. To integrate geospatial predictors into a harmonised geospatial database for Nairobi.
2. To compare logistic regression, Random Forest, and XGBoost for binary flood-susceptibility modelling.
3. To quantify model performance and uncertainty under alternative sampling strategies and feature-selection choices.
4. To produce an interpretable flood-susceptibility surface and driver ranking to support flood-risk management in Nairobi.


### Methodology

This project follows the CRISP-DM framework: Business Understanding, Data Understanding (audit), Data Preparation (alignment, validation), Modeling (Random Forest), Evaluation, and Deployment (susceptibility map generation). The study used a harmonised raster dataset for Nairobi built from static geospatial layers and event-based flood observations. Static predictors included elevation (DEM), slope, built-up/urban cover, and baseline water/land-cover layers aligned to a common grid. For each flood event, dynamic rainfall layers (e.g., 7-day accumulation and event-period totals) were paired with a binary flood/water mask used as the target label. All rasters were resampled to the same projection and resolution so pixel-level samples could be extracted consistently for model training and mapping.

## Section 1: Data Understanding and Overview 


Before using the data, we check for missing files, corrupted rasters, excessive nodata coverage, and constant values that indicate data issues.

In [ ]:
import os
import glob
import re
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_bounds
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.base import clone
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.ensemble import StackingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.metrics import brier_score_loss, roc_auc_score, precision_score, recall_score, f1_score
from sklearn.calibration import calibration_curve


In [ ]:
DATA_DIR = 'data/nairobi_flood_data'  # Extracted from nairobi_flood_data.zip

print(f"Scanning data directory: {DATA_DIR}\n")

# Check if the static rasters are present
static_patterns = {
    'dem': '*dem*nairobi*.tif',
    'slope': '*slope*nairobi*.tif',
    'builtup': '*builtup*nairobi*.tif',
    'landcover': '*landcover*nairobi*.tif',
    'perm_water': '*perm_water*nairobi*.tif'
}

static_rasters = {}
for name, pattern in static_patterns.items():
    matches = glob.glob(os.path.join(DATA_DIR, pattern))
    if matches:
        static_rasters[name] = matches[0]
        print(f"✓ Found {name}: {os.path.basename(static_rasters[name])}")
    else:
        print(f"Missing {name} (pattern: {pattern})")

# Check critical static rasters
if 'dem' not in static_rasters:
    raise FileNotFoundError(f"DEM raster not found in {DATA_DIR}. Cannot proceed without reference raster.")

# Discover event IDs from label files
label_pattern = os.path.join(DATA_DIR, 'label_event_water_*.tif')
label_files = glob.glob(label_pattern)

event_ids = []
for lf in label_files:
    # Extract event ID: label_event_water_E??_YYYY_MM*.tif -> E??_YYYY_MM*
    basename = os.path.basename(lf)
    match = re.search(r'(E\d+_\d{4}_\d{2}[a-zA-Z]*)', basename)
    if match:
        event_ids.append(match.group(1))

print(f"\n✓ Found {len(event_ids)} event labels: {event_ids}")

if len(event_ids) == 0:
    raise FileNotFoundError(f"No event labels found in {DATA_DIR}. Check file naming: label_event_water_E??_YYYY_MM*.tif")

# Validate events: check if all required files exist
events_data = []
for event_id in event_ids:
    event_record = {'event_id': event_id}
    
    # Check label
    label_matches = glob.glob(os.path.join(DATA_DIR, f'label_event_water_{event_id}.tif'))
    event_record['label'] = label_matches[0] if label_matches else None
    
    # Check rain7
    rain7_matches = glob.glob(os.path.join(DATA_DIR, f'rain7_{event_id}.tif'))
    event_record['rain7'] = rain7_matches[0] if rain7_matches else None
    
    # Check rain_sum
    rain_sum_matches = glob.glob(os.path.join(DATA_DIR, f'rain_sum_{event_id}.tif'))
    event_record['rain_sum'] = rain_sum_matches[0] if rain_sum_matches else None
    
    # Validate completeness
    missing = []
    if not event_record['label']:
        missing.append('label')
    if not event_record['rain7']:
        missing.append('rain7')
    if not event_record['rain_sum']:
        missing.append('rain_sum')
    
    event_record['complete'] = len(missing) == 0
    event_record['missing_files'] = ','.join(missing) if missing else ''
    
    events_data.append(event_record)

events_df = pd.DataFrame(events_data)

# Summary
print("\n" + "="*70)
print("DATASET DISCOVERY SUMMARY")
print("="*70)
print(f"Static rasters found: {len(static_rasters)}/{len(static_patterns)}")
print(f"Events discovered: {len(event_ids)}")
print(f"Complete events: {events_df['complete'].sum()}")
print(f"Incomplete events: {(~events_df['complete']).sum()}")

if (~events_df['complete']).any():
    print("\n⚠ Dropping incomplete events:")
    incomplete = events_df[~events_df['complete']]
    for _, row in incomplete.iterrows():
        print(f"  - {row['event_id']}: missing {row['missing_files']}")

valid_events = events_df[events_df['complete']]

if len(valid_events) == 0:
    raise ValueError("No valid events found after validation. Check that events have all required files (label, rain7, rain_sum).")

print(f"\n✓ Keeping {len(valid_events)} valid events: {list(valid_events['event_id'])}")

# Build CONFIG
CONFIG = {
    'data_dir': DATA_DIR,
    'static_features': ['dem', 'slope', 'builtup'],  # Drop landcover for now (categorical)
    'event_features': ['rain7', 'rain_sum'],
    'static_rasters': static_rasters,
    'events': valid_events.to_dict('records'),
    'reference_raster': static_rasters['dem'],  # Use DEM as reference
    'target_resolution': 30,
    'samples_per_event': 200,  # Balanced: 200 flood + 200 non-flood per event
    'test_size': 0.25,  # Split by events
    'random_state': 42,
    'aligned_dir': 'data/processed/aligned',
    'outputs_dir': 'outputs',
}

os.makedirs(CONFIG['aligned_dir'], exist_ok=True)
os.makedirs(CONFIG['outputs_dir'], exist_ok=True)

print(f"\n✓ Configuration ready")
print(f"  Static features: {CONFIG['static_features']}")
print(f"  Event features: {CONFIG['event_features']}")
print(f"  Reference raster: {os.path.basename(CONFIG['reference_raster'])}")

- The data folder was checked and all required files were found.
- 5 static raster files were found: DEM, slope, built-up, land cover, and permanent water.
- 15 flood event label files were found. All events have complete data.
- No files are missing.

- Static features: DEM, slope, built-up area
- Event features: 7-day rainfall, event rainfall sum
- Reference raster: DEM (dem_srtm_30m_nairobi.tif)

This means the dataset is complete and ready for analysis.

In [ ]:
def audit_raster(name, path):
    """Check single raster file quality"""
    record = {'name': name, 'path': path}
    
    if not os.path.exists(path):
        record.update({'exists': False, 'warnings': 'FILE_NOT_FOUND'})
        return record
    
    record['exists'] = True
    record['file_size_kb'] = round(os.path.getsize(path) / 1024, 2)
    
    try:
        with rasterio.open(path) as src:
            record['width'] = src.width
            record['height'] = src.height
            record['crs'] = str(src.crs)
            
            data = src.read(1)
            nodata = src.nodata if src.nodata is not None else -9999
            
            valid_mask = data != nodata
            pct_nodata = 100 * (~valid_mask).sum() / data.size
            record['pct_nodata'] = round(pct_nodata, 2)
            
            valid_data = data[valid_mask]
            if len(valid_data) > 0:
                record['min'] = float(valid_data.min())
                record['max'] = float(valid_data.max())
                record['n_unique'] = len(np.unique(valid_data))
                
            warnings = []
            if record['file_size_kb'] < 5:
                warnings.append('TINY_FILE')
            if pct_nodata > 50:
                warnings.append('HIGH_NODATA')
            if len(valid_data) > 0 and valid_data.min() == valid_data.max():
                warnings.append('CONSTANT_VALUES')
                
            record['warnings'] = ','.join(warnings) if warnings else ''
            
    except Exception as e:
        record['warnings'] = f'READ_ERROR:{str(e)}'
    
    return record



In [ ]:
audit_results = []

# Audit static rasters
for name, path in CONFIG['static_rasters'].items():
    result = audit_raster(name, path)
    audit_results.append(result)

# Audit event rasters
for event in CONFIG['events']:
    event_id = event['event_id']
    for ftype in ['label', 'rain7', 'rain_sum']:
        path = event[ftype]
        result = audit_raster(f"{event_id}_{ftype}", path)
        audit_results.append(result)

audit_df = pd.DataFrame(audit_results)

print(f"Audited {len(audit_df)} rasters")
print(f"  Exists: {audit_df['exists'].sum()}")
print(f"  Issues: {(audit_df['warnings'] != '').sum()}")

In [ ]:
display(audit_df[['name', 'exists', 'width', 'height', 'pct_nodata', 'warnings']])


- All 50 raster files are present with no missing data.  
- Each file has the same size and no NoData values or warnings.



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))  # 1 row, 2 columns

# File Availability
axes[0].bar(['Exists', 'Missing'],
            [len(audit_df[audit_df['exists'] == True]), len(audit_df[audit_df['exists'] == False])],
            color=['green', 'red'])
axes[0].set_title('File Availability', fontweight='bold')
axes[0].set_ylabel('Count')

# NoData Coverage
axes[1].hist(audit_df[audit_df['exists'] == True]['pct_nodata'], bins=10, color='steelblue', edgecolor='black')
axes[1].set_title('NoData Coverage', fontweight='bold')
axes[1].set_xlabel('% NoData')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

All files present and readable. No significant data quality issues detected, as 99.8% of each raster contains valid data. Proceeding to alignment phase.



## **Section 2: Data Preparation - Raster Alignment**

### Alignment Purpose

All rasters must share the same coordinate system, extent, resolution, and pixel grid to enable pixel-by-pixel analysis. We align all rasters to match the DEM reference grid at 30m resolution.

In [ ]:
with rasterio.open(CONFIG['reference_raster']) as ref:
    ref_crs = ref.crs
    ref_bounds = ref.bounds
    ref_transform = ref.transform
    ref_width = ref.width
    ref_height = ref.height
    ref_nodata = ref.nodata if ref.nodata is not None else -9999

print(f"Reference Grid:")
print(f"  Dimensions: {ref_width} x {ref_height} pixels")
print(f"  CRS: {ref_crs}")
print(f"  Bounds: {ref_bounds}")
print(f"  NoData: {ref_nodata}")

In [ ]:
target_transform = from_bounds(
    ref_bounds.left, ref_bounds.bottom,
    ref_bounds.right, ref_bounds.top,
    ref_width, ref_height
)

print(f"✓ Target transform calculated")
print(f"  Pixel size: {abs(target_transform.a):.6f} x {abs(target_transform.e):.6f}")
print(f"\nAll rasters will be aligned to this grid")

In [ ]:
aligned_static = {}

for name in CONFIG['static_features']:
    if name not in CONFIG['static_rasters']:
        print(f"⚠ Skipping {name} (not found)")
        continue
        
    input_path = CONFIG['static_rasters'][name]
    output_path = os.path.join(CONFIG['aligned_dir'], f"{name}_aligned.tif")
    
    with rasterio.open(input_path) as src:
        # Determine resampling method
        resampling = Resampling.nearest if name in ['builtup', 'landcover'] else Resampling.bilinear
        
        # Get appropriate nodata value for this raster's dtype
        if src.nodata is not None:
            src_nodata = src.nodata
        else:
            # Choose dtype-appropriate nodata
            dtype = src.dtypes[0]
            if 'uint8' in dtype:
                src_nodata = 255
            elif 'uint16' in dtype:
                src_nodata = 65535
            elif 'int' in dtype or 'float' in dtype:
                src_nodata = -9999
            else:
                src_nodata = 0
        
        # Create output array
        aligned_data = np.empty((ref_height, ref_width), dtype=src.dtypes[0])
        
        # Reproject
        reproject(
            source=rasterio.band(src, 1),
            destination=aligned_data,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=target_transform,
            dst_crs=ref_crs,
            resampling=resampling,
            src_nodata=src_nodata,
            dst_nodata=src_nodata
        )
        
        # Save
        profile = src.profile.copy()
        profile.update({
            'transform': target_transform,
            'width': ref_width,
            'height': ref_height,
            'crs': ref_crs,
            'nodata': src_nodata,
            'compress': 'lzw'
        })
        
        with rasterio.open(output_path, 'w', **profile) as dst:
            dst.write(aligned_data, 1)
    
    aligned_static[name] = output_path
    print(f"  ✓ {name}")

print(f"\n✓ Aligned {len(aligned_static)} static rasters")

In [ ]:
fig, axes = plt.subplots(1, len(aligned_static), figsize=(5*len(aligned_static), 4))
if len(aligned_static) == 1:
    axes = [axes]

for idx, (fname, fpath) in enumerate(aligned_static.items()):
    with rasterio.open(fpath) as src:
        data = src.read(1)
        display_data = np.where(data == ref_nodata, np.nan, data)
        
        im = axes[idx].imshow(display_data, cmap='viridis', interpolation='nearest')
        axes[idx].set_title(f'{fname}', fontweight='bold')
        axes[idx].axis('off')
        plt.colorbar(im, ax=axes[idx], fraction=0.046, pad=0.04)

plt.suptitle('Aligned Static Rasters (30m Grid)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['outputs_dir'], 'aligned_static.png'), dpi=300, bbox_inches='tight')
plt.show()

Each static raster is aligned to the same 30m grid for Nairobi. This ensures all layers match perfectly for pixel-by-pixel analysis.

To ensure accurate pixel-by-pixel analysis across all flood events, each event raster (labels and rainfall) must be perfectly aligned to the same reference grid as the static layers. This step guarantees that every pixel location corresponds spatially across all datasets, preventing misalignment errors during modeling.

In [ ]:
aligned_events = []


for event in CONFIG['events']:
    event_id = event['event_id']
    aligned_event = {'event_id': event_id}
    
    print(f"Event: {event_id}")
    
    for ftype in ['label', 'rain7', 'rain_sum']:
        input_path = event[ftype]
        output_path = os.path.join(CONFIG['aligned_dir'], f"{event_id}_{ftype}_aligned.tif")
        
        with rasterio.open(input_path) as src:
            # Use nearest neighbor for labels, bilinear for rainfall
            resampling = Resampling.nearest if ftype == 'label' else Resampling.bilinear
            
            # Get appropriate nodata value for this raster's dtype
            if src.nodata is not None:
                src_nodata = src.nodata
            else:
                # Choose dtype-appropriate nodata
                dtype = src.dtypes[0]
                if 'uint8' in dtype:
                    src_nodata = 255
                elif 'uint16' in dtype:
                    src_nodata = 65535
                elif 'int' in dtype or 'float' in dtype:
                    src_nodata = -9999
                else:
                    src_nodata = 0
            
            aligned_data = np.empty((ref_height, ref_width), dtype=src.dtypes[0])
            
            reproject(
                source=rasterio.band(src, 1),
                destination=aligned_data,
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=target_transform,
                dst_crs=ref_crs,
                resampling=resampling,
                src_nodata=src_nodata,
                dst_nodata=src_nodata
            )
            
            profile = src.profile.copy()
            profile.update({
                'transform': target_transform,
                'width': ref_width,
                'height': ref_height,
                'crs': ref_crs,
                'nodata': src_nodata,
                'compress': 'lzw'
            })
            
            with rasterio.open(output_path, 'w', **profile) as dst:
                dst.write(aligned_data, 1)
        
        aligned_event[ftype] = output_path
        print(f"  ✓ {ftype}")
    
    aligned_events.append(aligned_event)

print(f"\n✓ Aligned rasters for {len(aligned_events)} events")

All rasters now share a common 30m grid covering the Nairobi study area.


## **Section 3: Data Preparation and Exploratory Data Analysis**

### 3.1 Per-Event Validation

### Validation Strategy

Each event must be validated independently to ensure:
1. Labels are binary (0/1 only, excluding nodata)
2. Sufficient positive pixels exist for training

3. Rainfall data is not all nodata or zeros4. The label represents transient flooding (not permanent water)

In [ ]:
def validate_event(event_data):
    """Validate a single event's data quality"""
    event_id = event_data['event_id']
    validation = {'event_id': event_id, 'valid': True, 'issues': []}
    
    # Load label
    with rasterio.open(event_data['label']) as src:
        label = src.read(1)
        label_nodata = src.nodata if src.nodata is not None else 255
    
    # Check binary
    label_mask = label != label_nodata
    unique_vals = np.unique(label[label_mask])
    if not np.all(np.isin(unique_vals, [0, 1])):
        validation['valid'] = False
        validation['issues'].append(f"Non-binary values: {unique_vals}")
    
    # Count positives
    n_positive = (label == 1).sum()
    validation['n_positive'] = int(n_positive)
    validation['pct_positive'] = round(100 * n_positive / label_mask.sum(), 4)
    
    if n_positive < 50:
        validation['valid'] = False
        validation['issues'].append(f"Too few positives: {n_positive}")
    
    # Load and check rain7 - HANDLES NaN VALUES
    with rasterio.open(event_data['rain7']) as src:
        rain7 = src.read(1)
        rain7_nodata = src.nodata if src.nodata is not None else -9999
    
    # Filter out NoData AND NaN values
    rain7_valid = rain7[np.isfinite(rain7) & (rain7 != rain7_nodata)]
    if len(rain7_valid) > 0:
        validation['rain7_min'] = float(rain7_valid.min())
        validation['rain7_max'] = float(rain7_valid.max())
    else:
        validation['rain7_min'] = None
        validation['rain7_max'] = None
    
    if len(rain7_valid) == 0 or (len(rain7_valid) > 0 and rain7_valid.max() <= 0):
        validation['valid'] = False
        validation['issues'].append("rain7 all nodata or zeros")
    
    # Load and check rain_sum - HANDLES NaN VALUES
    with rasterio.open(event_data['rain_sum']) as src:
        rain_sum = src.read(1)
        rain_sum_nodata = src.nodata if src.nodata is not None else -9999
    
    # Filter out NoData AND NaN values
    rain_sum_valid = rain_sum[np.isfinite(rain_sum) & (rain_sum != rain_sum_nodata)]
    if len(rain_sum_valid) > 0:
        validation['rain_sum_min'] = float(rain_sum_valid.min())
        validation['rain_sum_max'] = float(rain_sum_valid.max())
    else:
        validation['rain_sum_min'] = None
        validation['rain_sum_max'] = None
    
    if len(rain_sum_valid) == 0 or (len(rain_sum_valid) > 0 and rain_sum_valid.max() <= 0):
        validation['valid'] = False
        validation['issues'].append("rain_sum all nodata or zeros")
    
    validation['issues_str'] = '; '.join(validation['issues']) if validation['issues'] else ''
    
    return validation

# Re-run validation with the NaN-corrected function
validation_results = []
for event in aligned_events:
    result = validate_event(event)
    validation_results.append(result)

validation_df = pd.DataFrame(validation_results)
print(f"\n✓ FINAL validation results:")
print(f"  Valid events: {validation_df['valid'].sum()}")
print(f"  Events with rainfall data: {validation_df['rain7_max'].notna().sum()}")

# Display the updated table
display(validation_df[['event_id', 'valid', 'n_positive', 'pct_positive', 'rain7_max', 'rain_sum_max', 'issues_str']])

All 15 events passed validation with no missing or invalid data, and each event has unique rainfall statistics and sufficient positive flood pixels. The dataset is complete and ready for modeling, with no quality issues detected in any event.

In [ ]:
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.bar(validation_df['event_id'], validation_df['n_positive'], color='steelblue')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.ylabel('Positive Pixels')
plt.title('Flood Pixels per Event', fontweight='bold')
plt.grid(axis='y', alpha=0.3)

plt.subplot(1, 2, 2)
plt.bar(validation_df['event_id'], validation_df['pct_positive'], color='coral')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.ylabel('% Positive')
plt.title('Flood Percentage per Event', fontweight='bold')
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['outputs_dir'], 'event_validation.png'), dpi=300, bbox_inches='tight')
plt.show()


Events validated for data quality. No invalid events were found hence none were dropped. 

**Important Note:** Labels represent "event water excluding permanent water" as a proxy for flood extent. These are not field-verified flood extents but derived from Sentinel-1 water detection.

### 3.2 Exploratory Data Analysis (EDA)

Before modeling, we conduct comprehensive EDA to understand data distributions, relationships between variables, and identify potential patterns that could inform feature engineering and model selection.

In [ ]:
# Load sample data from multiple events for EDA
print("Loading sample data from events for EDA...")

# Load static rasters for EDA
static_eda_data = {}
for fname in CONFIG['static_features']:
    with rasterio.open(aligned_static[fname]) as src:
        data = src.read(1)
        # Remove nodata values
        valid_data = data[data != ref_nodata]
        static_eda_data[fname] = valid_data
        print(f"  ✓ {fname}: {len(valid_data):,} valid pixels")

# Load event data for first few events
event_eda_data = {}
sample_events = aligned_events[:5]  # First 5 events for EDA

for event in sample_events:
    event_id = event['event_id']
    event_eda_data[event_id] = {}
    
    for ftype in CONFIG['event_features']:
        with rasterio.open(event[ftype]) as src:
            data = src.read(1)
            # Remove nodata and NaN values
            valid_data = data[(data != -9999) & np.isfinite(data)]
            event_eda_data[event_id][ftype] = valid_data
    
    # Load labels
    with rasterio.open(event['label']) as src:
        label_data = src.read(1)
        label_nodata = src.nodata if src.nodata is not None else 255
        valid_labels = label_data[label_data != label_nodata]
        event_eda_data[event_id]['label'] = valid_labels
    




In [ ]:
crucial_features = ['dem', 'slope', 'builtup']
fig, axes = plt.subplots(1, len(crucial_features), figsize=(5*len(crucial_features), 5))

for idx, fname in enumerate(crucial_features):
    data = static_eda_data[fname]
    axes[idx].hist(data, bins=50, alpha=0.7, color='steelblue', edgecolor='black')
    axes[idx].set_title(f'{fname.upper()} Distribution', fontweight='bold')
    axes[idx].set_xlabel('Value')
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(alpha=0.3)

plt.suptitle('Key Static Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['outputs_dir'], 'eda_static_histograms_key.png'), dpi=300, bbox_inches='tight')
plt.show()

The univariate analysis plots show that elevation (DEM) and slope are right-skewed, reflecting Nairobi’s topography with most areas at lower elevations and gentle slopes. The built-up feature is strongly bimodal, indicating a clear distinction between urban and non-urban areas in the dataset.

In [ ]:
# Event Features Analysis
print("\n=== EVENT FEATURES ANALYSIS ===")

# Analyze rainfall patterns across events
for fname in CONFIG['event_features']:
    print(f"\n{fname.upper()} Analysis:")
    
    # Collect data from all sample events
    all_values = []
    event_stats = []
    
    for event_id, event_data in event_eda_data.items():
        if fname in event_data:
            values = event_data[fname]
            all_values.extend(values)
            mean_val = np.mean(values)
            max_val = np.max(values)
            event_stats.append({
                'event_id': event_id,
                'mean': mean_val,
                'max': max_val,
                'count': len(values)
            })
            print(f"  {event_id}: Mean={mean_val:.2f}mm, Max={max_val:.2f}mm")
    
    if all_values and len(event_stats) > 1:
        all_values = np.array(all_values)
        print(f"\nOverall {fname.upper()} Statistics:")
        print(f"  Mean: {np.mean(all_values):.4f}")
        print(f"  Std: {np.std(all_values):.4f}")
        print(f"  Min: {np.min(all_values):.4f}")
        print(f"  Max: {np.max(all_values):.4f}")
        
        # Simple visualization
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        
        # Distribution
        ax1.hist(all_values, bins=30, alpha=0.7, color='green', edgecolor='black')
        ax1.set_title(f'{fname.upper()} Distribution', fontweight='bold')
        ax1.set_xlabel('Value (mm)')
        ax1.set_ylabel('Frequency')
        ax1.grid(alpha=0.3)
        
        # Event comparison
        event_names = [stat['event_id'][-8:] for stat in event_stats]
        event_means = [stat['mean'] for stat in event_stats]
        
        ax2.bar(range(len(event_names)), event_means, color='orange', alpha=0.7)
        ax2.set_title(f'{fname.upper()} by Event', fontweight='bold')
        ax2.set_xlabel('Event')
        ax2.set_ylabel('Mean Value (mm)')
        ax2.set_xticks(range(len(event_names)))
        ax2.set_xticklabels(event_names, rotation=45)
        ax2.grid(alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(os.path.join(CONFIG['outputs_dir'], f'eda_{fname}_analysis.png'), dpi=300, bbox_inches='tight')
        plt.show()

The rainfall feature analysis shows substantial variability in both 7-day cumulative rainfall (RAIN7) and event rainfall sum (RAIN_SUM) across different flood events. RAIN7 values range from very low (mean 2.6 mm) to very high (mean 190.7 mm), with an overall mean of 90.3 mm and a standard deviation of 67.0 mm. RAIN_SUM also varies widely, with event means between 151.5 mm and 292.7 mm, and an overall mean of 212.0 mm. These distributions confirm that the dataset captures a broad spectrum of rainfall intensities, which is essential for robust flood susceptibility modeling.



In [ ]:
# Correlation Analysis (Full Modeling Dataset)
print("\n=== CORRELATION ANALYSIS (FULL DATASET) ===")

# Use the full modeling dataset for correlation analysis
feature_names = CONFIG['static_features'] + CONFIG['event_features']

if 'dataset' in globals() and len(dataset) > 100:
    # Compute correlation matrix
    corr_matrix = dataset[feature_names + ['label']].corr()
    
    # Create correlation heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, 
                annot=True, 
                cmap='RdBu_r', 
                center=0,
                square=True, 
                fmt='.3f',
                cbar_kws={'label': 'Correlation Coefficient'},
                linewidths=0.5)
    plt.title('Feature Correlation Matrix (Full Dataset)', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['outputs_dir'], 'eda_correlation_matrix_full.png'), dpi=300, bbox_inches='tight')
    plt.show()
    
    # Print correlations with flood label
    print("\nCorrelations with Flood Occurrence:")
    label_corrs = corr_matrix['label'].drop('label').sort_values(key=abs, ascending=False)
    for feature, corr_val in label_corrs.items():
        direction = "positive" if corr_val > 0 else "negative"
        strength = "strong" if abs(corr_val) > 0.3 else "moderate" if abs(corr_val) > 0.1 else "weak"
        print(f"  {feature}: {corr_val:+.4f} ({strength} {direction})")
    
    # Feature intercorrelations
    print("\nStrongest Feature Intercorrelations:")
    feature_corr_matrix = corr_matrix.loc[feature_names, feature_names]
    strong_corrs = []
    for i in range(len(feature_names)):
        for j in range(i+1, len(feature_names)):
            feat1, feat2 = feature_names[i], feature_names[j]
            corr_val = feature_corr_matrix.loc[feat1, feat2]
            if abs(corr_val) > 0.2:  # Show correlations > 0.2
                strong_corrs.append((feat1, feat2, corr_val))
    strong_corrs.sort(key=lambda x: abs(x[2]), reverse=True)
    for feat1, feat2, corr_val in strong_corrs[:5]:  # Top 5
        print(f"  {feat1} <-> {feat2}: {corr_val:+.4f}")
else:
    print("Full modeling dataset not available or too small. Please run the dataset building section first.")

The correlation matrix shows that none of the features have a strong linear relationship with flood occurrence, as all correlations with the flood label are weak and negative. However, there are strong intercorrelations among predictors, especially between elevation (dem) and rain_sum (0.84), and between rain7 and rain_sum (0.57), indicating some redundancy among rainfall and terrain variables.

---

## Section 4: Modeling - Build Training Dataset 

Sample pixels separately for each event, attach `event_id`, then split by event to ensure test events are completely unseen during training.

In [ ]:
# Load static rasters (shared across all events)
print("Loading static rasters...\n")

static_arrays = {}
for fname in CONFIG['static_features']:
    with rasterio.open(aligned_static[fname]) as src:
        static_arrays[fname] = src.read(1)
        print(f"  ✓ {fname}: shape {static_arrays[fname].shape}")

print(f"\n✓ Static features loaded: {list(static_arrays.keys())}")

In [ ]:
def sample_event(event_data, static_arrays, k_samples=200):
    """Sample pixels from a single event (balanced)"""
    event_id = event_data['event_id']
    
    # Load event rasters
    with rasterio.open(event_data['label']) as src:
        label = src.read(1)
        label_nodata = src.nodata if src.nodata is not None else 255
    
    event_features = {}
    for fname in CONFIG['event_features']:
        with rasterio.open(event_data[fname]) as src:
            event_features[fname] = src.read(1)
    
    # Build valid mask (no nodata in any raster)
    valid_mask = label != label_nodata
    
    for arr in static_arrays.values():
        valid_mask &= (arr != ref_nodata)
    
    for arr in event_features.values():
        valid_mask &= (arr != -9999)  # Assume rain nodata is -9999
    
    # Separate flood vs non-flood
    flood_mask = valid_mask & (label == 1)
    noflood_mask = valid_mask & (label == 0)
    
    n_flood = flood_mask.sum()
    n_noflood = noflood_mask.sum()
    
    # Sample (balanced)
    k_flood = min(k_samples, n_flood)
    k_noflood = min(k_samples, n_noflood)
    
    if k_flood == 0:
        print(f"  ⚠ {event_id}: no flood pixels available")
        return None
    
    # Random sample indices
    np.random.seed(CONFIG['random_state'] + hash(event_id) % 10000)
    
    flood_rows, flood_cols = np.where(flood_mask)
    flood_idx = np.random.choice(len(flood_rows), size=k_flood, replace=False)
    
    noflood_rows, noflood_cols = np.where(noflood_mask)
    noflood_idx = np.random.choice(len(noflood_rows), size=k_noflood, replace=False)
    
    # Extract features
    sample_data = {'event_id': []}
    for fname in CONFIG['static_features']:
        sample_data[fname] = []
    for fname in CONFIG['event_features']:
        sample_data[fname] = []
    sample_data['label'] = []
    
    # Add flood samples
    for i in flood_idx:
        sample_data['event_id'].append(event_id)
        for fname in CONFIG['static_features']:
            sample_data[fname].append(static_arrays[fname][flood_rows[i], flood_cols[i]])
        for fname in CONFIG['event_features']:
            sample_data[fname].append(event_features[fname][flood_rows[i], flood_cols[i]])
        sample_data['label'].append(1)
    
    # Add non-flood samples
    for i in noflood_idx:
        sample_data['event_id'].append(event_id)
        for fname in CONFIG['static_features']:
            sample_data[fname].append(static_arrays[fname][noflood_rows[i], noflood_cols[i]])
        for fname in CONFIG['event_features']:
            sample_data[fname].append(event_features[fname][noflood_rows[i], noflood_cols[i]])
        sample_data['label'].append(0)
    
    df = pd.DataFrame(sample_data)
    
    print(f"  ✓ {event_id}: sampled {k_flood} flood + {k_noflood} non-flood = {len(df)} total")
    
    return df


In [ ]:
# Filter to valid events only
validated_events = [event for event in aligned_events 
                   if event['event_id'] in validation_df[validation_df['valid']]['event_id'].values]

print(f"Sampling from {len(validated_events)} events...\n")

event_dfs = []
for event in validated_events:
    df = sample_event(event, static_arrays, k_samples=CONFIG['samples_per_event'])
    if df is not None:
        event_dfs.append(df)

# Combine all events
dataset = pd.concat(event_dfs, ignore_index=True)

print(f"\n✓ Combined dataset shape: {dataset.shape}")
print(f"  Total samples: {len(dataset):,}")
print(f"  Events: {dataset['event_id'].nunique()}")

In [ ]:
print("Samples per event:")
print(dataset.groupby('event_id').size().sort_values(ascending=False))

In [ ]:
print("Class balance:")
print(dataset.groupby(['event_id', 'label']).size().unstack(fill_value=0))

print(f"\nOverall class balance:")
print(dataset['label'].value_counts())
print(f"\nSample data:")
display(dataset.head(10))

In [ ]:
print("Feature statistics:")
feature_cols = CONFIG['static_features'] + CONFIG['event_features']
display(dataset[feature_cols].describe())

In [ ]:
feature_cols = CONFIG['static_features'] + CONFIG['event_features']
n_features = len(feature_cols)

fig, axes = plt.subplots(2, (n_features + 1) // 2, figsize=(15, 8))
axes = axes.flatten()

for idx, fname in enumerate(feature_cols):
    axes[idx].hist(dataset[fname], bins=50, color='steelblue', alpha=0.7, edgecolor='black')
    axes[idx].set_title(f'{fname}', fontweight='bold')
    axes[idx].set_xlabel('Value')
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(alpha=0.3)

# Class distribution in last subplot
axes[n_features].bar(['No Flood', 'Flood'], dataset['label'].value_counts().sort_index(), color=['lightcoral', 'dodgerblue'])
axes[n_features].set_title('Class Distribution', fontweight='bold')
axes[n_features].set_ylabel('Count')
axes[n_features].grid(alpha=0.3)

# Hide extra subplots
for idx in range(n_features + 1, len(axes)):
    axes[idx].axis('off')

plt.suptitle('Training Dataset Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['outputs_dir'], 'dataset_distributions.png'), dpi=300, bbox_inches='tight')
plt.show()

The feature distribution plots show that the training dataset covers a wide range of values for each predictor, indicating good variability for model learning. Elevation (dem) and slope are both right-skewed, reflecting Nairobi’s topography with most areas at lower elevations and gentle slopes, but with some higher terrain and steeper areas. The built-up feature is strongly bimodal, as expected for a binary urban mask, with most samples being non-urban and a smaller proportion urban. Rainfall features (rain7 and rain_sum) display broad, positively skewed distributions, confirming that the sampled events span a variety of rainfall intensities and totals. The class distribution plot demonstrates perfect balance between flood and non-flood samples, which is ideal for unbiased model training. Overall, these distributions confirm that the dataset is well-prepared for robust flood susceptibility modeling, with no evidence of missing data or sampling bias.




Per-event sampling complete. Each sample is tagged with `event_id` to enable proper train/test splitting without data leakage. Class balance is maintained per event.

---

## Section 5: Modeling - Train and Evaluate (Event-Based Split)

Split by `event_id` so test events are completely unseen during training. This provides realistic evaluation of how the model generalizes to new flood events.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# Split by event IDs
unique_events = dataset['event_id'].unique()
train_events, test_events = train_test_split(
    unique_events, 
    test_size=CONFIG['test_size'], 
    random_state=CONFIG['random_state']
)

print(f"Event-based split:")
print(f"  Train events ({len(train_events)}): {list(train_events)}")
print(f"  Test events ({len(test_events)}): {list(test_events)}")

# Create train/test datasets
train_mask = dataset['event_id'].isin(train_events)
test_mask = dataset['event_id'].isin(test_events)

feature_cols = CONFIG['static_features'] + CONFIG['event_features']

X_train = dataset.loc[train_mask, feature_cols]
y_train = dataset.loc[train_mask, 'label']

X_test = dataset.loc[test_mask, feature_cols]
y_test = dataset.loc[test_mask, 'label']

print(f"\nTrain set: {len(y_train):,} samples")
print(f"Test set: {len(y_test):,} samples")
print(f"\nFeatures: {feature_cols}")

# Check for missing/invalid values
print(f"\n--- Data Quality Check ---")
print(f"Train set NaN count: {X_train.isna().sum().sum()}")
print(f"Train set inf count: {np.isinf(X_train.values).sum()}")
print(f"Test set NaN count: {X_test.isna().sum().sum()}")
print(f"Test set inf count: {np.isinf(X_test.values).sum()}")

# Remove samples with NaN or inf values
if X_train.isna().any().any() or np.isinf(X_train.values).any():
    print(f"\n⚠ Cleaning training data...")
    train_valid_mask = ~(X_train.isna().any(axis=1) | pd.DataFrame(np.isinf(X_train.values), index=X_train.index).any(axis=1))
    X_train = X_train[train_valid_mask]
    y_train = y_train[train_valid_mask]
    print(f"  Removed {(~train_valid_mask).sum()} samples with NaN/inf")
    print(f"  Remaining: {len(X_train):,} samples")

if X_test.isna().any().any() or np.isinf(X_test.values).any():
    print(f"\n⚠ Cleaning test data...")
    test_valid_mask = ~(X_test.isna().any(axis=1) | pd.DataFrame(np.isinf(X_test.values), index=X_test.index).any(axis=1))
    X_test = X_test[test_valid_mask]
    y_test = y_test[test_valid_mask]
    print(f"  Removed {(~test_valid_mask).sum()} samples with NaN/inf")
    print(f"  Remaining: {len(X_test):,} samples")

print(f"\n✓ Data ready for training")

In [ ]:


models = {}

print("Training models...\n")

# 1. Logistic Regression
print("1. Logistic Regression")
lr = LogisticRegression(
    max_iter=1000,
    random_state=CONFIG['random_state'],
    n_jobs=-1
)
lr.fit(X_train, y_train)
models['Logistic Regression'] = lr
print("   ✓ Trained\n")

# 2. Random Forest
print("2. Random Forest")
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=CONFIG['random_state'],
    n_jobs=-1,
    verbose=0
)
rf.fit(X_train, y_train)
models['Random Forest'] = rf
print("   ✓ Trained\n")

# 3. XGBoost
print("3. XGBoost")
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=CONFIG['random_state'],
    n_jobs=-1,
    eval_metric='logloss'
)
xgb.fit(X_train, y_train)
models['XGBoost'] = xgb
print("   ✓ Trained\n")

# 4. Ensemble (Voting)
print("4. Ensemble (Average Probabilities)")
print("   ✓ Will combine predictions from all three models\n")

print(f"✓ All {len(models)} base models trained")

In [ ]:
# Stacking (OOF meta-features from RF & XGBoost) -> Logistic Regression meta-learner


# Base learners (match your training hyperparams if you changed them earlier)
base_rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
base_xgb = XGBClassifier(n_estimators=200, use_label_encoder=False, eval_metric='logloss', random_state=42, n_jobs=-1)

# Meta-learner: regularized logistic to penalize overconfident outputs (L2 via C)
meta_clf = LogisticRegression(C=0.5, solver='lbfgs', max_iter=2000, random_state=42)

# OOF stacking
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_rf = np.zeros(len(X_train))
oof_xgb = np.zeros(len(X_train))

for train_idx, val_idx in skf.split(X_train, y_train):
    trX, trY = X_train.iloc[train_idx], y_train.iloc[train_idx]
    vX = X_train.iloc[val_idx]

    m_rf = clone(base_rf)
    m_xgb = clone(base_xgb)

    m_rf.fit(trX, trY)
    m_xgb.fit(trX, trY)

    oof_rf[val_idx] = m_rf.predict_proba(vX)[:, 1]
    oof_xgb[val_idx] = m_xgb.predict_proba(vX)[:, 1]

# Train meta-learner on OOF meta-features
meta_X = np.vstack([oof_rf, oof_xgb]).T
meta_clf.fit(meta_X, y_train)

# Retrain base learners on full training data (for test-time meta-features)
base_rf.fit(X_train, y_train)
base_xgb.fit(X_train, y_train)

test_meta = np.vstack([
    base_rf.predict_proba(X_test)[:, 1],
    base_xgb.predict_proba(X_test)[:, 1]
]).T

# Stacked predictions
ensemble_proba = meta_clf.predict_proba(test_meta)[:, 1]
ensemble_pred = (ensemble_proba >= 0.5).astype(int)

print("Stacking ensemble (RF+XGB -> LR meta) on test:")
print(f"  Samples: {len(y_test):,}")
print(f"  Accuracy: {accuracy_score(y_test, ensemble_pred):.4f}")
print(f"  ROC AUC:  {roc_auc_score(y_test, ensemble_proba):.4f}")

In [ ]:


def summarize(name, y_true, proba, pred):
    print(f"{name}: Acc={np.mean(pred==y_true):.4f}, ROC_AUC={roc_auc_score(y_true, proba):.4f}, Brier={brier_score_loss(y_true, proba):.4f}")
    p, r = precision_score(y_true, pred), recall_score(y_true, pred)
    print(f"       Precision={p:.4f}, Recall={r:.4f}, F1={f1_score(y_true, pred):.4f}")

# existing average ensemble (if kept)
avg_proba = np.mean([
    models['Random Forest'].predict_proba(X_test)[:,1],
    models['XGBoost'].predict_proba(X_test)[:,1],
    models['Logistic Regression'].predict_proba(X_test)[:,1]
], axis=0)
avg_pred = (avg_proba >= 0.5).astype(int)

summarize("Averaging Ensemble", y_test.values, avg_proba, avg_pred)
summarize("Stacking Ensemble", y_test.values, ensemble_proba, ensemble_pred)

# calibration curve (stacking)
prob_true, prob_pred = calibration_curve(y_test, ensemble_proba, n_bins=10)
print("Stacking calibration (first 3 bins):", prob_true[:3], prob_pred[:3])

In [ ]:

print("meta coef:", getattr(meta_clf, "coef_", None))
print("meta intercept:", getattr(meta_clf, "intercept_", None))
print("OOF lengths:", len(oof_rf), len(oof_xgb), len(y_train))

In [ ]:
rf_p = base_rf.predict_proba(X_test)[:,1]
xgb_p = base_xgb.predict_proba(X_test)[:,1]
mean_base = (rf_p + xgb_p) / 2
print("MAE(stack, mean_base):", np.mean(np.abs(ensemble_proba - mean_base)))
import numpy as np
print("corrs: rf-xgb", np.corrcoef(rf_p, xgb_p)[0,1], "stack-mean", np.corrcoef(ensemble_proba, mean_base)[0,1])

In [ ]:

for C in [0.01,0.1,0.5,1,5,10]:
    m = LogisticRegression(C=C, max_iter=2000, random_state=42).fit(meta_X, y_train)
    test_meta = np.vstack([base_rf.predict_proba(X_test)[:,1], base_xgb.predict_proba(X_test)[:,1]]).T
    print("C",C,"ROC_AUC",roc_auc_score(y_test, m.predict_proba(test_meta)[:,1]))

In [ ]:
cal_rf = CalibratedClassifierCV(base_rf, cv=5)
cal_xgb = CalibratedClassifierCV(base_xgb, cv=5)
cal_rf.fit(X_train,y_train); cal_xgb.fit(X_train,y_train)

stack = StackingClassifier(
    estimators=[('rf', cal_rf), ('xgb', cal_xgb)],
    final_estimator=LogisticRegression(C=0.1, max_iter=2000),
    cv=5, passthrough=True, n_jobs=-1
)
stack.fit(X_train,y_train)
stack_proba = stack.predict_proba(X_test)[:,1]
print("StackingClassifier passthrough ROC_AUC:", roc_auc_score(y_test, stack_proba))

In [ ]:


results = []

print("Evaluating models on test set...\n")

for model_name, model in models.items():
    print(f"Evaluating {model_name}...")
    
    # Predictions
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    # Metrics
    metrics = {
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'ROC_AUC': roc_auc_score(y_test, y_proba)
    }
    
    results.append(metrics)
    
    print(f"  Accuracy:  {metrics['Accuracy']:.4f}")
    print(f"  Precision: {metrics['Precision']:.4f}")
    print(f"  Recall:    {metrics['Recall']:.4f}")
    print(f"  F1:        {metrics['F1']:.4f}")
    print(f"  ROC AUC:   {metrics['ROC_AUC']:.4f}\n")

# Ensemble predictions (average probabilities)
print("Evaluating Ensemble (Average)...")
ensemble_proba = np.mean([
    models['Logistic Regression'].predict_proba(X_test)[:, 1],
    models['Random Forest'].predict_proba(X_test)[:, 1],
    models['XGBoost'].predict_proba(X_test)[:, 1]
], axis=0)

ensemble_pred = (ensemble_proba >= 0.5).astype(int)

ensemble_metrics = {
    'Model': 'Ensemble',
    'Accuracy': accuracy_score(y_test, ensemble_pred),
    'Precision': precision_score(y_test, ensemble_pred),
    'Recall': recall_score(y_test, ensemble_pred),
    'F1': f1_score(y_test, ensemble_pred),
    'ROC_AUC': roc_auc_score(y_test, ensemble_proba)
}

results.append(ensemble_metrics)

print(f"  Accuracy:  {ensemble_metrics['Accuracy']:.4f}")
print(f"  Precision: {ensemble_metrics['Precision']:.4f}")
print(f"  Recall:    {ensemble_metrics['Recall']:.4f}")
print(f"  F1:        {ensemble_metrics['F1']:.4f}")
print(f"  ROC AUC:   {ensemble_metrics['ROC_AUC']:.4f}\n")

results_df = pd.DataFrame(results)
print("✓ All models evaluated")

In [ ]:
# Display results table
print("\n" + "="*80)
print("MODEL COMPARISON - TEST SET PERFORMANCE")
print("="*80)
display(results_df.sort_values('ROC_AUC', ascending=False))

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC_AUC']
colors = ['steelblue', 'coral', 'seagreen', 'mediumpurple']

for idx, metric in enumerate(metrics_to_plot):
    row = idx // 3
    col = idx % 3
    
    ax = axes[row, col]
    bars = ax.bar(results_df['Model'], results_df[metric], color=colors)
    ax.set_ylabel(metric.replace('_', ' '), fontsize=11)
    ax.set_title(f'{metric.replace("_", " ")}', fontweight='bold', fontsize=12)
    ax.set_ylim([0, 1])
    ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis='x', rotation=45, labelsize=9)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontsize=9)

# Summary in last subplot
ax = axes[1, 2]
ax.axis('off')
best_model = results_df.loc[results_df['ROC_AUC'].idxmax(), 'Model']
best_auc = results_df.loc[results_df['ROC_AUC'].idxmax(), 'ROC_AUC']
summary_text = f"""
MODEL COMPARISON SUMMARY

Best Model (by ROC AUC):
{best_model}
ROC AUC = {best_auc:.4f}

Total Models Trained: 4
- Logistic Regression
- Random Forest  
- XGBoost
- Ensemble (Average)

Evaluation on {len(y_test):,} test samples
from {len(test_events)} unseen events
"""
ax.text(0.1, 0.5, summary_text, fontsize=11, verticalalignment='center',
        family='monospace', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

plt.suptitle('Model Performance Comparison - Test Set', fontsize=15, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['outputs_dir'], 'model_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()

This plots compares the performance of four models—Logistic Regression, Random Forest, XGBoost, and an Ensemble—on unseen test flood events. The evaluation metrics include accuracy, precision, recall, F1 score, and ROC AUC. XGBoost achieved the highest ROC AUC of 0.915, indicating its superior ability to distinguish flood-prone areas. The Ensemble model closely followed, leveraging combined predictions for balanced performance. Logistic Regression performed the weakest, highlighting its limitations in capturing complex flood patterns. Tree-based models like Random Forest and XGBoost excelled in handling non-linear interactions between terrain, rainfall, and land cover features. This analysis identifies XGBoost as the best model for flood susceptibility mapping.

In [ ]:
# Confusion matrices for all models
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

model_names = ['Logistic Regression', 'Random Forest', 'XGBoost', 'Ensemble']

for idx, model_name in enumerate(model_names):
    if model_name == 'Ensemble':
        y_pred = ensemble_pred
    else:
        y_pred = models[model_name].predict(X_test)
    
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
                xticklabels=['No Flood', 'Flood'],
                yticklabels=['No Flood', 'Flood'],
                ax=axes[idx])
    axes[idx].set_ylabel('Actual Label', fontsize=11)
    axes[idx].set_xlabel('Predicted Label', fontsize=11)
    axes[idx].set_title(f'{model_name}', fontsize=12, fontweight='bold')

plt.suptitle('Confusion Matrices - Test Set', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['outputs_dir'], 'confusion_matrices_all.png'), dpi=300, bbox_inches='tight')
plt.show()

The confusion matrices compare the performance of four models on the test set. Each matrix shows the number of true positives, true negatives, false positives, and false negatives for flood and no-flood predictions. XGBoost and Ensemble models demonstrate better balance, with fewer false positives and false negatives compared to Logistic Regression and Random Forest. Logistic Regression struggles with misclassifying no-flood areas as flood-prone, while Random Forest has higher false negatives for flood predictions. XGBoost and Ensemble models provide more accurate predictions, making them suitable for flood susceptibility mapping.


In [ ]:
from sklearn.metrics import roc_curve, auc

# ROC curves for all models
plt.figure(figsize=(10, 8))

colors_roc = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
model_names_with_ensemble = ['Logistic Regression', 'Random Forest', 'XGBoost', 'Ensemble']

for idx, model_name in enumerate(model_names_with_ensemble):
    if model_name == 'Ensemble':
        y_proba = ensemble_proba
    else:
        y_proba = models[model_name].predict_proba(X_test)[:, 1]
    
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    
    plt.plot(fpr, tpr, color=colors_roc[idx], lw=2.5, 
             label=f'{model_name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves - All Models (Test Set)', fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['outputs_dir'], 'roc_curves_all.png'), dpi=300, bbox_inches='tight')
plt.show()

The ROC curves compare the ability of four models—Logistic Regression, Random Forest, XGBoost, and Ensemble—to distinguish flood-prone areas, with XGBoost achieving the highest AUC of 0.915. The Ensemble model closely follows, demonstrating strong predictive performance, while Logistic Regression shows the weakest results.


In [ ]:
# Feature importance comparison (for tree-based models)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Random Forest
rf_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': models['Random Forest'].feature_importances_
}).sort_values('importance', ascending=True)

axes[0].barh(rf_importance['feature'], rf_importance['importance'], color='steelblue')
axes[0].set_xlabel('Importance Score', fontsize=11)
axes[0].set_ylabel('Feature', fontsize=11)
axes[0].set_title('Random Forest Feature Importance', fontsize=12, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

# XGBoost
xgb_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': models['XGBoost'].feature_importances_
}).sort_values('importance', ascending=True)

axes[1].barh(xgb_importance['feature'], xgb_importance['importance'], color='seagreen')
axes[1].set_xlabel('Importance Score', fontsize=11)
axes[1].set_ylabel('Feature', fontsize=11)
axes[1].set_title('XGBoost Feature Importance', fontsize=12, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['outputs_dir'], 'feature_importance_comparison.png'), 
            dpi=300, bbox_inches='tight')
plt.show()

print("\nFeature Importance Rankings:\n")
print("Random Forest:")
display(rf_importance.sort_values('importance', ascending=False))
print("\nXGBoost:")
display(xgb_importance.sort_values('importance', ascending=False))

The feature importance plots show the relative contribution of predictors to flood susceptibility modeling for Random Forest and XGBoost. Random Forest emphasizes terrain features like elevation (DEM) and slope, while XGBoost prioritizes built-up areas, followed by elevation, indicating its focus on urbanization patterns in flood risk prediction.



In [ ]:
# Select best model for deployment
best_model_name = results_df.loc[results_df['ROC_AUC'].idxmax(), 'Model']
print(f"Best performing model: {best_model_name}")
print(f"ROC AUC: {results_df.loc[results_df['ROC_AUC'].idxmax(), 'ROC_AUC']:.4f}")

if best_model_name == 'Ensemble':
    print("\n✓ Using Ensemble model for predictions")
    best_model = None  # Will use ensemble_proba calculation
else:
    best_model = models[best_model_name]
    print(f"\n✓ Using {best_model_name} for predictions")


Four models were trained and evaluated on unseen test events:

1. **Logistic Regression**: Linear baseline model
2. **Random Forest**: Ensemble of decision trees
3. **XGBoost**: Gradient boosting with regularization
4. **Ensemble**: Average of all three model probabilities

Event-based splitting ensures realistic evaluation - the models have never seen these flood events during training. The best performing model will be used for generating the susceptibility map.

## Section 6: Deployment 

## Prediction Strategy - https://nairobifloodmapping.streamlit.app/

Choose one event for prediction and apply the trained model to all pixels in the study area using that event's rainfall data. The output shows flood susceptibility given those rainfall conditions.

In [ ]:
import joblib

# Save all models
model_package = {
    'models': models,
    'best_model_name': best_model_name,
    'ensemble_available': True,
    'features': feature_cols,
    'config': CONFIG,
    'train_events': list(train_events),
    'test_events': list(test_events),
    'results': results_df
}

model_path = os.path.join(CONFIG['outputs_dir'], 'all_models.pkl')
joblib.dump(model_package, model_path)

print(f"✓ All models saved to: {model_path}")
print(f"  Models included: {list(models.keys())}")
print(f"  Best model: {best_model_name}")
print(f"  Features: {feature_cols}")
print(f"  Train events: {list(train_events)}")
print(f"  Test events: {list(test_events)}")

In [ ]:
# Choose event for prediction
prediction_event = validated_events[-1]
prediction_event_id = prediction_event['event_id']

print(f"Generating susceptibility map for event: {prediction_event_id}")
print(f"Using best model: {best_model_name}\n")

# Load models
pkg = joblib.load(model_path)
loaded_models = pkg['models']
feature_order = pkg['features']

print(f"✓ Models loaded")
print(f"  Feature order: {feature_order}")

In [ ]:
print(f"Loading prediction rasters for {prediction_event_id}...\n")

pred_stack = []

# Load static features
for fname in CONFIG['static_features']:
    with rasterio.open(aligned_static[fname]) as src:
        pred_stack.append(src.read(1))
        profile = src.profile.copy()
        print(f"  ✓ {fname}")

# Load event features for the chosen event
for fname in CONFIG['event_features']:
    with rasterio.open(prediction_event[fname]) as src:
        pred_stack.append(src.read(1))
        print(f"  ✓ {fname} ({prediction_event_id})")

pred_stack = np.stack(pred_stack)
print(f"\nPrediction stack shape: {pred_stack.shape}")

In [ ]:
pred_mask = np.ones(pred_stack.shape[1:], dtype=bool)

for i in range(pred_stack.shape[0]):
    pred_mask &= (pred_stack[i] != ref_nodata) & (pred_stack[i] != -9999)

n_valid = pred_mask.sum()
n_total = pred_mask.size

print(f"Valid pixels to predict:")
print(f"  {n_valid:,} / {n_total:,} ({100*n_valid/n_total:.1f}%)")

In [ ]:
output = np.full(pred_stack.shape[1:], -9999.0, dtype=np.float32)
valid_rows, valid_cols = np.where(pred_mask)

print(f"Initialized output array: {output.shape}")

In [ ]:
chunk_size = 50000
n_valid_pixels = len(valid_rows)

print(f"Predicting in chunks of {chunk_size:,}...\n")

for start in range(0, n_valid_pixels, chunk_size):
    end = min(start + chunk_size, n_valid_pixels)
    rows = valid_rows[start:end]
    cols = valid_cols[start:end]
    
    # Extract features in correct order
    X_chunk = np.column_stack([pred_stack[i, rows, cols] 
                                for i in range(len(feature_order))])
    
    # Predict probabilities using best model
    if best_model_name == 'Ensemble':
        # Calculate ensemble prediction
        proba = np.mean([
            loaded_models['Logistic Regression'].predict_proba(X_chunk)[:, 1],
            loaded_models['Random Forest'].predict_proba(X_chunk)[:, 1],
            loaded_models['XGBoost'].predict_proba(X_chunk)[:, 1]
        ], axis=0)
    else:
        proba = loaded_models[best_model_name].predict_proba(X_chunk)[:, 1]
    
    output[rows, cols] = proba
    
    if (end % (chunk_size * 5) == 0) or (end == n_valid_pixels):
        print(f"  Progress: {end:,} / {n_valid_pixels:,} ({100*end/n_valid_pixels:.1f}%)")

In [ ]:
output_path = os.path.join(CONFIG['outputs_dir'], 
                           f"flood_susceptibility_{best_model_name.replace(' ', '_')}_{prediction_event_id}.tif")

profile.update(dtype=rasterio.float32, nodata=-9999.0, compress='lzw')

with rasterio.open(output_path, 'w', **profile) as dst:
    dst.write(output, 1)

print(f"\n✓ Susceptibility map saved to: {output_path}")
print(f"  Model: {best_model_name}")
print(f"  Event: {prediction_event_id}")

In [ ]:
valid_pred = output[output != -9999.0]

print(f"\nPrediction Statistics:")
print(f"  Min:    {valid_pred.min():.4f}")
print(f"  Max:    {valid_pred.max():.4f}")
print(f"  Mean:   {valid_pred.mean():.4f}")
print(f"  Median: {np.median(valid_pred):.4f}")
print(f"  Std:    {valid_pred.std():.4f}")

In [ ]:
plt.figure(figsize=(12, 10))

display_output = np.where(output == -9999.0, np.nan, output)

plt.imshow(display_output, cmap='RdYlBu_r', vmin=0, vmax=1, interpolation='nearest')
cbar = plt.colorbar(label='Flood Susceptibility (0=Low, 1=High)', shrink=0.8)
plt.title(f'Flood Susceptibility Map - {best_model_name}\nNairobi (Event: {prediction_event_id})', 
          fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['outputs_dir'], f'susceptibility_map_{best_model_name.replace(" ", "_")}.png'), 
            dpi=300, bbox_inches='tight')
plt.show()

The output map shows spatial patterns of flood risk for the selected event's rainfall conditions generated using the XGBoost model. High susceptibility areas (red) concentrate along expected flood-prone zones. This map is specific to the rainfall characteristics of the chosen event and represents the best performing model among the four tested approaches.

---

## Section 7: Discussion and Conclusions

### Key Findings

The event-based modeling approach successfully identifies flood-prone areas using terrain, rainfall, and land cover features. Four machine learning models were trained and evaluated:

1. Logistic Regression: Provides interpretable baseline with linear relationships
2. Random Forest: Captures non-linear interactions through ensemble of trees
3. XGBoost: Gradient boosting with regularization for improved generalization
4. Ensemble: Combines all three models through probability averaging

Event-based train/test splitting provides realistic performance estimates for how well the models generalize to new flood events. The comparison reveals which algorithm best captures flood susceptibility patterns in Nairobi.

### **Limitations**

- Label quality: Labels are derived from Sentinel-1 water detection, not field-verified flood extents. They represent "event water" as a proxy for flooding.

- Limited events: Results depend on the number and diversity of flood events available. More events would improve model robustness.

- Spatial resolution: 30m resolution may miss localized flooding in small catchments or under dense vegetation.

- Temporal representation: Model captures flood patterns from available events but may not generalize to different climate conditions or extreme events outside the training distribution.

- Data leakage prevention: Event-based splitting is essential but may show lower performance than pixel-based random splitting. This is expected and represents realistic generalization ability.

### **Recommendations**

1. Employ the XGBoost model for flood susceptibility mapping: Based on its high accuracy (ROC AUC = 0.915), XGBoost is recommended for operational use in identifying flood-prone areas in Nairobi. Its ability to capture complex interactions between terrain, rainfall, and urbanization makes it suitable for proactive disaster management and infrastructure planning.

2. Focus on urbanization and terrain in flood mitigation strategies: The analysis highlights the importance of built-up areas and elevation in flood susceptibility. Stakeholders should prioritize improving drainage systems in urbanized regions and implementing terrain-based interventions, such as reforestation or slope stabilization, to reduce flood risks.

3. Expand research to other flood-prone regions in Kenya: Apply this methodology to other vulnerable areas across the country, such as Kisumu or Mombasa, to develop region-specific flood susceptibility maps. This will support nationwide disaster preparedness and infrastructure planning efforts.

### Conclusions

This project aimed to develop a flood susceptibility mapping framework for Nairobi, Kenya, using satellite data and machine learning techniques. The analysis successfully identified high-risk flood zones by leveraging terrain, rainfall, and urbanization features. A robust workflow was implemented, including data validation, raster alignment, event-based sampling, and model evaluation to ensure accuracy and reliability. Four machine learning models were trained and compared, with XGBoost achieving the highest predictive performance (ROC AUC = 0.915). Deliverables included a validated training dataset, susceptibility maps, and a reproducible methodology for future applications. This framework provides a scalable solution for flood risk assessment and can be adapted to other flood-prone regions globally.